# Day 13 — Data Validation, Custom Exceptions & Bad Records

Modules: `src/schema.py`, `src/validator13.py`  
Dataset: Olist Brazilian E-Commerce

In [1]:
# Parameters
INPUT_FOLDER       = "../data"
BAD_RECORDS_PATH   = "../data/bad_records.csv"
REJECTION_THRESHOLD = 0.20

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

%load_ext autoreload
%autoreload 2

## Task 1 — Required fields per file (defined in schema.py)

In [3]:
from schema import SCHEMA, ALLOWED_ORDER_STATUSES

print("Required fields per file:")
for fname, fields in SCHEMA.items():
    print(f"  {fname}: {fields}")

print(f"\nAllowed order statuses: {ALLOWED_ORDER_STATUSES}")

Required fields per file:
  orders.csv: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_estimated_delivery_date']
  order_items.csv: ['order_id', 'product_id', 'seller_id', 'price', 'freight_value', 'shipping_limit_date']
  customers.csv: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix']
  products.csv: ['product_id']
  sellers.csv: ['seller_id', 'seller_zip_code_prefix']
  order_payments.csv: ['order_id', 'payment_value']
  order_reviews.csv: ['review_id', 'order_id']

Allowed order statuses: {'canceled', 'invoiced', 'approved', 'shipped', 'processing', 'delivered', 'unavailable', 'created'}


## Task 7 — Custom exceptions: SchemaError and FileFormatError

In [4]:
from schema import SchemaError, FileFormatError

print(f"SchemaError    base class: {SchemaError.__bases__}")
print(f"FileFormatError base class: {FileFormatError.__bases__}")

SchemaError    base class: (<class 'Exception'>,)
FileFormatError base class: (<class 'Exception'>,)


## Task 8 — Handle missing or empty file gracefully

In [5]:
from validator13 import load_file_safe

for test_path in ["../data/nonexistent.csv", "../data/orders.csv"]:
    try:
        rows = load_file_safe(test_path)
        print(f"OK: {test_path} loaded ({len(rows)} rows)")
    except FileFormatError as e:
        print(f"FileFormatError: {e}")

FileFormatError: File not found: ../data/nonexistent.csv
OK: ../data/orders.csv loaded (502 rows)


## Tasks 2, 4, 5, 6 — Validate rows, check types, business rules, regex

In [6]:
from validator13 import validate_row

# Good row
good = {
    "order_id":                     "a" * 32,
    "customer_id":                  "b" * 32,
    "order_status":                 "delivered",
    "order_purchase_timestamp":     "2021-01-01 10:00:00",
    "order_estimated_delivery_date":"2021-01-10 00:00:00",
}
print("Good row errors:", validate_row(good, list(good.keys()), "orders.csv", 2))

# Bad row — wrong status, bad date, bad ID
bad = {
    "order_id":                     "not-valid",
    "customer_id":                  "b" * 32,
    "order_status":                 "unknown",
    "order_purchase_timestamp":     "not-a-date",
    "order_estimated_delivery_date":"2021-01-10 00:00:00",
}
print("\nBad row errors:")
for e in validate_row(bad, list(bad.keys()), "orders.csv", 3):
    print(" ", e)

Good row errors: []

Bad row errors:
  row 3: 'order_purchase_timestamp' is not a valid datetime (got 'not-a-date')
  row 3: 'order_status' invalid (got 'unknown')
  row 3: 'order_id' must be 32 hex chars (got 'not-valid')


## Tasks 3, 9, 10 — Validate full file, write bad_records.csv, print summary, fail on threshold

In [7]:
import os
from validator13 import validate_file

files_to_validate = [
    ("orders.csv",      SCHEMA["orders.csv"]),
    ("order_items.csv", SCHEMA["order_items.csv"]),
    ("customers.csv",   SCHEMA["customers.csv"]),
]

for fname, required in files_to_validate:
    filepath = os.path.join(INPUT_FOLDER, fname)
    bad_path = os.path.join(INPUT_FOLDER, f"bad_{fname}")
    try:
        summary = validate_file(
            filepath,
            required,
            bad_records_path=bad_path,
            rejection_threshold=REJECTION_THRESHOLD,
        )
        print(f"\n{summary['file']}")
        print(f"  Total   : {summary['total']}")
        print(f"  Valid   : {summary['valid']}")
        print(f"  Rejected: {summary['rejected']}")
        print(f"  Rate    : {summary['rejection_rate']:.1%}")
        if summary['rejected'] > 0:
            print(f"  Bad records written to: {bad_path}")
    except SchemaError as e:
        print(f"\nSCHEMA ERROR — {e}")
    except FileFormatError as e:
        print(f"\nFILE ERROR — {e}")


orders.csv
  Total   : 502
  Valid   : 500
  Rejected: 2
  Rate    : 0.4%
  Bad records written to: ../data\bad_orders.csv

order_items.csv
  Total   : 576
  Valid   : 576
  Rejected: 0
  Rate    : 0.0%

customers.csv
  Total   : 500
  Valid   : 500
  Rejected: 0
  Rate    : 0.0%


## Run pytest

In [8]:
!python -m pytest ../tests/test_day13.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.10, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\Arun\AppData\Local\Programs\Python\Python314\python.exe
cachedir: .pytest_cache
rootdir: D:\Arun\Tasks\python-git
collecting ... collected 10 items

..\tests\test_day13.py::test_valid_order_row_returns_no_errors PASSED    [ 10%]
..\tests\test_day13.py::test_missing_required_field PASSED               [ 20%]
..\tests\test_day13.py::test_invalid_order_status PASSED                 [ 30%]
..\tests\test_day13.py::test_price_not_numeric PASSED                    [ 40%]
..\tests\test_day13.py::test_price_zero_fails PASSED                     [ 50%]
..\tests\test_day13.py::test_invalid_zip_prefix PASSED                   [ 60%]
..\tests\test_day13.py::test_invalid_hex_id PASSED                       [ 70%]
..\tests\test_day13.py::test_load_file_safe_missing_raises PASSED        [ 80%]
..\tests\test_day13.py::test_load_file_safe_empty_raises P